# 165. Tensor Parallel：Column/Row Linear、通信与 vocab-parallel loss 怎样实现？

> **面试问题：张量并行为什么常把 MLP 的第一层按列切、第二层按行切？All-Gather/All-Reduce 到底在哪里？**

## 先给结论

Tensor Parallel 的核心不是“把矩阵切开”一句话，而是让切分轴与下一算子的输入布局配合：Column Parallel 产生分片输出，逐分片激活后直接喂给 Row Parallel，后者对局部部分和做 All-Reduce。回答还应覆盖不整除切分、vocab-parallel 交叉熵、反向等价、通信量与 checkpoint 重分片。

## 推荐的回答主线

1. 从 `Y=XW` 的轴出发，分别推导按 W 输出列切和按 W 输入行切的局部结果。
2. 组合两层 MLP，说明中间激活为何可保持分片以及何时需要 collective。
3. 用 full reference 比较 forward、loss 和 gradient，而不是只看形状能拼起来。
4. 把通信字节、拓扑、序列并行、容错和跨 world-size checkpoint 纳入工程回答。

## 本 Notebook 的实现边界

代码在单进程模拟 rank 与 collective，证明代数和布局；没有调用 NCCL，也不代表实际重叠、链路竞争或 kernel 效率。生产性能需在目标 mesh、dtype 和消息大小上 profiling。

## 一手资料

- [Megatron-LM](https://arxiv.org/abs/1909.08053)
- [Efficient Large-Scale Language Model Training](https://arxiv.org/abs/2104.04473)
- [PyTorch Tensor Parallel API](https://docs.pytorch.org/docs/stable/distributed.tensor.parallel.html)


In [ ]:
import hashlib
import json
import math
from dataclasses import dataclass, asdict

import numpy as np
import torch

# 单进程把列表元素当作各 rank 的局部张量，便于逐项检查 collective 语义。
rng = np.random.default_rng(165)
BATCH, IN_DIM, HIDDEN, OUT_DIM, WORLD = 5, 7, 11, 6, 3
x_np = rng.normal(size=(BATCH, IN_DIM))
w1_np = rng.normal(size=(IN_DIM, HIDDEN))
w2_np = rng.normal(size=(HIDDEN, OUT_DIM))

assert x_np.shape[1] == w1_np.shape[0]
assert w1_np.shape[1] == w2_np.shape[0]
assert HIDDEN % WORLD != 0  # 故意覆盖不整除分片。


## 1. 平衡切片：生产代码不能默认维度可整除

`array_split` 风格的平衡区间让前若干 rank 多拿一个元素。所有 rank 必须共享同一布局元数据；否则 forward 可能在 concat 时看似正确，checkpoint 或反向通信却发生错位。


In [ ]:
def balanced_slices(size, world):
    base, extra = divmod(size, world)
    result, start = [], 0
    for rank in range(world):
        width = base + (rank < extra)
        result.append(slice(start, start + width))
        start += width
    return result

# 切片必须无缝覆盖全轴，任意两片不重叠，宽度至多相差 1。
hidden_slices = balanced_slices(HIDDEN, WORLD)
widths = [sl.stop - sl.start for sl in hidden_slices]
assert hidden_slices[0].start == 0 and hidden_slices[-1].stop == HIDDEN
assert sum(widths) == HIDDEN
assert max(widths) - min(widths) <= 1


## 2. Column Parallel Linear：切 W 的输出列，再 All-Gather

每个 rank 都持有完整 X 和一部分 `W[:, j0:j1]`，因此得到对应输出 feature 分片。若下一层能消费分片，可以不立即 gather；这里 concat 只用于对照完整 GEMM。


In [ ]:
def column_parallel(x, weight, world, gather=True):
    shards = [x @ weight[:, sl] for sl in balanced_slices(weight.shape[1], world)]
    return np.concatenate(shards, axis=1) if gather else shards

# 拼接后的结果必须与完整矩阵乘逐元素相等，局部宽度匹配布局。
column_shards = column_parallel(x_np, w1_np, WORLD, gather=False)
column_full = np.concatenate(column_shards, axis=1)
assert np.allclose(column_full, x_np @ w1_np)
assert [part.shape[1] for part in column_shards] == widths
assert all(part.shape[0] == BATCH for part in column_shards)


## 3. Row Parallel Linear：切输入 feature，局部部分和再 All-Reduce

按 W 的输入行切时，X 必须用相同区间切 feature。每个 rank 产生完整输出形状的部分和；All-Reduce(sum) 后才是最终 Y。若误用 concat，形状或数值都会错。


In [ ]:
def row_parallel(x, weight, world):
    slices = balanced_slices(weight.shape[0], world)
    partials = [x[:, sl] @ weight[sl, :] for sl in slices]
    reduced = np.sum(np.stack(partials, axis=0), axis=0)
    return reduced, partials

# 所有局部结果形状相同，求和严格复原完整 Linear。
row_full, row_partials = row_parallel(rng.normal(size=(BATCH, HIDDEN)), w2_np, WORLD)
reference_input = np.concatenate([rng.normal(size=(1, 1))], axis=0)  # 独立占位，避免隐藏状态复用误导。
test_hidden = np.concatenate(column_shards, axis=1)
row_from_hidden, partials_from_hidden = row_parallel(test_hidden, w2_np, WORLD)
assert row_from_hidden.shape == (BATCH, OUT_DIM)
assert np.allclose(row_from_hidden, test_hidden @ w2_np)
assert all(part.shape == (BATCH, OUT_DIM) for part in partials_from_hidden)


## 4. Column → GELU → Row：中间激活无需 All-Gather

逐元素 GELU 不跨 hidden feature，因此可在各 Column shard 本地计算；第二层 W2 按相同 hidden 区间切行，局部输出求和即可。这正是 Megatron MLP 常用布局，减少中间 collective。


In [ ]:
def gelu(z):
    return 0.5 * z * (1.0 + np.tanh(math.sqrt(2.0 / math.pi) * (z + 0.044715 * z**3)))

def tp_mlp(x, w1, w2, world):
    slices = balanced_slices(w1.shape[1], world)
    local_hidden = [gelu(x @ w1[:, sl]) for sl in slices]
    local_output = [hidden @ w2[sl, :] for hidden, sl in zip(local_hidden, slices)]
    return np.sum(local_output, axis=0), local_hidden

# 分片 MLP 与完整参考必须一致，中间各片总宽度等于 hidden size。
tp_output, local_hidden = tp_mlp(x_np, w1_np, w2_np, WORLD)
full_output = gelu(x_np @ w1_np) @ w2_np
assert np.allclose(tp_output, full_output)
assert sum(part.shape[1] for part in local_hidden) == HIDDEN
assert len(local_hidden) == WORLD


## 5. Vocab-parallel 交叉熵：全局 max、sum 和目标 logit 都是 collective

词表 logits 按列分片后，稳定 softmax 需要先对局部最大值做全局 max，再对指数和做全局 sum；目标 token 的 logit 只在所属 rank 非零，最后 sum。少一步都会让 loss 随 world size 改变。


In [ ]:
def vocab_parallel_nll(logit_shards, targets, slices):
    local_max = [part.max(axis=1) for part in logit_shards]
    global_max = np.max(np.stack(local_max), axis=0)
    global_sum = sum(np.exp(part - global_max[:, None]).sum(axis=1) for part in logit_shards)
    target_logit = np.zeros(targets.shape[0])
    for part, sl in zip(logit_shards, slices):
        owned = (targets >= sl.start) & (targets < sl.stop)
        target_logit[owned] = part[owned, targets[owned] - sl.start]
    return np.log(global_sum) + global_max - target_logit

# 分片 NLL 应等于完整 log-sum-exp 参考，并在大 logits 下保持有限。
vocab_logits = rng.normal(size=(BATCH, 13)) * 20 + 1000
vocab_slices = balanced_slices(vocab_logits.shape[1], WORLD)
vocab_shards = [vocab_logits[:, sl] for sl in vocab_slices]
targets_np = np.array([0, 4, 7, 11, 12])
nll = vocab_parallel_nll(vocab_shards, targets_np, vocab_slices)
ref_nll = np.log(np.exp(vocab_logits - vocab_logits.max(1, keepdims=True)).sum(1)) + vocab_logits.max(1) - vocab_logits[np.arange(BATCH), targets_np]
assert np.allclose(nll, ref_nll)
assert np.isfinite(nll).all()
assert nll.shape == (BATCH,)


## 6. 反向 oracle：分片计算的输入与权重梯度也要等价

只验证 forward 不够。下面用 PyTorch autograd 对相同 leaf 权重构造分片 MLP，并与完整路径比较梯度。真实分布式实现由 collective 的 autograd 规则负责布局转换，但数学 oracle 不应改变。


In [ ]:
# 两条图使用独立但数值相同的叶子张量，避免梯度相互累加。
xt = torch.tensor(x_np, dtype=torch.float64, requires_grad=True)
w1t = torch.tensor(w1_np, dtype=torch.float64, requires_grad=True)
w2t = torch.tensor(w2_np, dtype=torch.float64, requires_grad=True)
ref = torch.nn.functional.gelu(xt @ w1t, approximate="tanh") @ w2t
ref.square().mean().backward()
ref_grads = (xt.grad.clone(), w1t.grad.clone(), w2t.grad.clone())

xs = torch.tensor(x_np, dtype=torch.float64, requires_grad=True)
w1s = torch.tensor(w1_np, dtype=torch.float64, requires_grad=True)
w2s = torch.tensor(w2_np, dtype=torch.float64, requires_grad=True)
local = [torch.nn.functional.gelu(xs @ w1s[:, sl], approximate="tanh") @ w2s[sl, :] for sl in hidden_slices]
sum(local).square().mean().backward()
assert torch.allclose(xs.grad, ref_grads[0], atol=1e-10)
assert torch.allclose(w1s.grad, ref_grads[1], atol=1e-10)
assert torch.allclose(w2s.grad, ref_grads[2], atol=1e-10)


## 7. 通信模型：字节数相同，拓扑与重叠也会改变尾延迟

Ring All-Reduce 每 rank 的理想通信量近似 `2*(p-1)/p*N` 字节；All-Gather 约 `(p-1)/p*N`。这是带宽项，不含每跳 latency、协议开销、拓扑跨域和与计算重叠。小消息往往由 latency 主导。


In [ ]:
def ring_collective_bytes(payload_bytes, world, kind):
    factor = (world - 1) / world
    if kind == "all_reduce":
        return 2 * factor * payload_bytes
    if kind == "all_gather":
        return factor * payload_bytes
    raise ValueError("未知 collective")

# world=1 无通信；同 payload 下 All-Reduce 是 All-Gather 的两倍带宽项。
payload = BATCH * OUT_DIM * 2  # 假设 FP16 输出。
assert ring_collective_bytes(payload, 1, "all_reduce") == 0
assert ring_collective_bytes(payload, WORLD, "all_reduce") == 2 * ring_collective_bytes(payload, WORLD, "all_gather")
assert ring_collective_bytes(payload, WORLD, "all_reduce") > 0


## 8. Checkpoint 重分片：保存全局轴范围，不依赖旧 rank 编号

弹性恢复时 world size 可能改变。制品应描述全局 shape、切分轴、每片范围、dtype 和权重摘要；恢复先验证全局张量，再按新布局重切。仅保存 `rank_1.pt` 文件名不足以证明语义。


In [ ]:
@dataclass(frozen=True)
class ShardManifest:
    global_shape: tuple
    axis: int
    ranges: tuple
    dtype: str
    sha256: str

def make_manifest(weight, world, axis):
    ranges = tuple((sl.start, sl.stop) for sl in balanced_slices(weight.shape[axis], world))
    digest = hashlib.sha256(weight.tobytes()).hexdigest()
    return ShardManifest(weight.shape, axis, ranges, str(weight.dtype), digest)

# 新旧 world size 的范围不同，但重组后的全局摘要必须相同。
manifest3 = make_manifest(w1_np, 3, 1)
manifest4 = make_manifest(w1_np, 4, 1)
assert manifest3.sha256 == manifest4.sha256
assert manifest3.ranges != manifest4.ranges
assert sum(end - start for start, end in manifest4.ranges) == HIDDEN


## 面试收束：怎样把实现讲成工程答案

建议按“目标与约束 → 数据/张量合同 → 核心公式 → 正确性 oracle → 性能与安全边界 → 发布门禁”作答。Notebook 里的小张量和受控状态机只证明机制成立，不等于真实集群吞吐、真实模型质量或生产安全性。上线前还要补齐目标硬件 profiling、故障注入、分布式一致性、真实数据切片、权限审计、版本化制品和回滚演练。

可继续追问：规模扩大后哪个状态最贵？哪条等价性可作为回归测试？输入或版本变化时怎样拒绝静默错误？指标改善是否只是成本、数据污染或评测器偏差造成的？
